#### Notebook for running React experiments

In [1]:
import sys, os
sys.path.append('..')
root  = '../root/'

In [2]:
os.environ['OPENAI_API_KEY']='xFKNcSuYHd2YZPzZxtMQrGR7AB49uwYA'
os.environ['LOCAL_OPENAI_BASE_URL']="https://ellm.nrp-nautilus.io/v1"
os.environ['MODEL_NAME']='gpt-oss'
os.environ['TEMPERATure']='0.7'

In [3]:
# from openai import OpenAI


In [4]:

# client = OpenAI(
#     openai_api_key="xFKNcSuYHd2YZPzZxtMQrGR7AB49uwYA",
#     openai_api_base="https://llm.nrp-nautilus.io/v1",
#     model_name='gpt-oss'
# )

# client = OpenAI(
#     # This is the default and can be omitted
#     api_key = os.environ.get("OPENAI_API_KEY"),
#     base_url = "https://ellm.nrp-nautilus.io/v1"
# )

# completion = client.chat.completions.create(
#     model="gpt-oss",
#     messages=[
#         {"role": "system", "content": "Talk like a pirate."},
#         {
#             "role": "user",
#             "content": "How do I check if a Python object is an instance of a class?",
#         },
#     ],
# )

# print(completion.choices[0].message.content)

In [5]:
print(os.environ['OPENAI_API_KEY'])

xFKNcSuYHd2YZPzZxtMQrGR7AB49uwYA


In [ ]:
import joblib
from util import summarize_react_trial, log_react_trial, save_agents
from agents import ReactReflectAgent, ReactAgent, ReflexionStrategy

#### Load the HotpotQA Sample

In [7]:
hotpot = joblib.load('../data/hotpot-qa-distractor-sample.joblib').reset_index(drop = True)

In [8]:
print(hotpot.iterrows())

<generator object DataFrame.iterrows at 0x1512f02e0>


#### Define the Reflexion Strategy

In [9]:
print(ReflexionStrategy.__doc__)

An enumeration.


In [10]:
strategy: ReflexionStrategy = ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION

#### Initialize a React Agent for each question

In [11]:
agent_cls = ParallelReactReflectAgent if strategy != ReflexionStrategy.NONE else ReactAgent
agents = [agent_cls(row['question'], row['answer']) for i, row in hotpot.iterrows() if i<1]

#### Run `n` trials

In [12]:
n = 1
trial = 0
log = ''

In [13]:
for i in range(n):
    for agent in [a for a in agents if not a.is_correct()]:
        if strategy != ReflexionStrategy.NONE:
            agent.run(reflect_strategy = strategy)
        else:
            agent.run()
        print(f'Answer: {agent.key}')
    trial += 1
    log += log_react_trial(agents, trial)
    correct, incorrect, halted = summarize_react_trial(agents)
    print(f'Finished Trial {trial}, Correct: {len(correct)}, Incorrect: {len(incorrect)}, Halted: {len(halted)}')

Thought 1: VIVA Media AG was renamed in 2004 to **VIVA Media Holding AG**, where the name “VIVA” is an acronym that stands for **“Vertriebs‑ und Informations‑Vermittlungs‑Agentur”**.
Thought 2: Vertriebs‑ und Informations‑Verbund‑Agentur.
Thought 3: VIVA Media Holding AG’s name is an acronym for “Vertriebs‑ und Informations‑Agentur”.
Thought 4: Action 1: Search[VIVA Media AG]
Observation 4 (Search[VIVA Media AG]): Could not find [VIVA Media AG]. Similar: ['MTV Music (Poland)', 'Paramount International Networks', 'VIVA Plus', 'VIVA Hungary', 'VIVA Germany', 'Viacom (1952–2005)', 'Vauxhall Viva', 'Sartorius AG', 'Spellbound Entertainment', 'Stephen Hawking']
Thought 5: Action 1a: Search[VIVA Media AG]
Observation 5a (Search[VIVA Media AG]): Could not find [VIVA Media AG]. Similar: ['MTV Music (Poland)', 'Paramount International Networks', 'VIVA Plus', 'VIVA Hungary', 'VIVA Germany', 'Viacom (1952–2005)', 'Vauxhall Viva', 'Sartorius AG', 'Spellbound Entertainment', 'Stephen Hawking']
Thou

#### Save the result log

In [14]:

# with open(os.path.join(root, 'ReAct', strategy.value, f'{len(agents)}_questions_{trial}_trials.txt'), 'w') as f:
#     f.write(log)
# # types that break pickling
# # types that break serialization
# BAD_TYPES = (
#     "_thread.RLock",
#     "_thread.lock",
#     "SSLContext",
#     "SSLSocket",
#     "CoreBPE",
# )

# def sanitize(obj, seen=None):
#     """Recursively null unpicklable runtime objects."""
#     if seen is None:
#         seen = set()
#     oid = id(obj)
#     if oid in seen:
#         return
#     seen.add(oid)

#     # If object itself is bad
#     if any(t in str(type(obj)) for t in BAD_TYPES):
#         return None

#     # Recurse into attributes
#     if hasattr(obj, "__dict__"):
#         for k, v in list(obj.__dict__.items()):
#             try:
#                 sanitized = sanitize(v, seen)
#                 if sanitized is None:
#                     setattr(obj, k, None)
#             except Exception:
#                 setattr(obj, k, None)

#     # Recurse into containers
#     elif isinstance(obj, dict):
#         for k, v in obj.items():
#             sanitized = sanitize(v, seen)
#             if sanitized is None:
#                 obj[k] = None
#     elif isinstance(obj, list):
#         for i, v in enumerate(obj):
#             sanitized = sanitize(v, seen)
#             if sanitized is None:
#                 obj[i] = None
#     elif isinstance(obj, tuple):
#         for i, v in enumerate(obj):
#             sanitized = sanitize(v, seen)
#             # tuples are immutable, skip if problematic
#     elif isinstance(obj, set):
#         to_remove = set()
#         for v in obj:
#             sanitized = sanitize(v, seen)
#             if sanitized is None:
#                 to_remove.add(v)
#         obj -= to_remove

#     return obj

# s_agents=[]
# for i, agent in enumerate(agents):
#         s= sanitize(agent) 
#         s_agents.append(s)
# save_agents(s_agents, os.path.join('ReAct', strategy.value, 'agents'))



In [15]:
print(agents[0].answer)

In [16]:
for agent in agents:
    agent.reflect_llm=None
    agent.react_llm=None
file_path = os.path.join(root, 'ReAct', strategy.value, f'{len(agents)}_questions_{trial}_trials.txt')
os.makedirs(os.path.dirname(file_path), exist_ok=True)
with open(file_path, 'w') as f:
    f.write(log)
save_agents(agents, os.path.join('ReAct', strategy.value, 'agents'))

TypeError: cannot pickle '_thread.RLock' object

In [ ]:
def rehydrate_agent(agent):
    """
    Restore non-picklable runtime components.
    Makes loaded agent executable again.
    """
    from llm import AnyOpenAILLM
    from langchain import Wikipedia
    from langchain.agents.react.base import DocstoreExplorer
    import tiktoken

    # ---- Restore LLMs ----
    if hasattr(agent, "react_llm"):
        agent.react_llm = AnyOpenAILLM()

    if hasattr(agent, "reflect_llm"):
        agent.reflect_llm = AnyOpenAILLM()

    # ---- Restore Wikipedia environment ----
    if hasattr(agent, "docstore"):
        agent.docstore = DocstoreExplorer(Wikipedia())

    # ---- Restore tokenizer ----
    if hasattr(agent, "enc"):
        agent.enc = tiktoken.encoding_for_model("text-davinci-003")

    return agent

In [ ]:
import os
import joblib

def load_agents(dir: str):
    agents = []

    files = sorted(
        f for f in os.listdir(dir)
        if f.endswith(".joblib")
    )

    for f in files:
        agent = joblib.load(os.path.join(dir, f))
        agent = rehydrate_agent(agent)
        agents.append(agent)

    return agents

In [ ]:
agents = load_agents(os.path.join('ReAct', strategy.value, 'agents'))

agents[0]

In [ ]:
for i in range(n):
    for agent in [a for a in agents if not a.is_correct()]:
        if strategy != ReflexionStrategy.NONE:
            agent.run(reflect_strategy = strategy)
        else:
            agent.run()
        print(f'Answer: {agent.key}')
    trial += 1
    log += log_react_trial(agents, trial)
    correct, incorrect, halted = summarize_react_trial(agents)
    print(f'Finished Trial {trial}, Correct: {len(correct)}, Incorrect: {len(incorrect)}, Halted: {len(halted)}')